### Missing values in the data

should they just be removed?

In [14]:
from datetime import datetime
import pandas as pd
import helper_functions_GNN as helper
from STGNN import STGNN
import numpy as np
import torch
import torch.nn as nn
import torch_geometric.nn as gnn
import matplotlib.pyplot as plt


BASE = "https://dashboard.elering.ee/api"
# Fetching data for 2019-2025
START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"

df_prices = helper.fetch_all(helper.get_nps_prices, START, END)
df_flows = helper.fetch_all(helper.get_cross_border_flows, START, END)
df_system = helper.fetch_all(helper.get_system_production, START, END)

# --- 3. standardize to hourly, then daily ---

df_prices_daily  = df_prices.resample("h").mean()
df_flows_daily   = df_flows.resample("h").mean()
df_system_daily  = df_system.resample("h").mean()

# --- 4. merge into one wide dataframe ---

df_daily = pd.concat([
    df_prices_daily.add_prefix("price_"),
    df_flows_daily.add_prefix("flow_"),
    df_system_daily.add_prefix("system_"),
], axis=1).sort_index()

print(f"\nMissing values:\n{df_daily.isna().sum()}")


  Fetching 2019-01-01 → 2019-02-01...
  Fetching 2019-02-01 → 2019-03-01...
  Fetching 2019-03-01 → 2019-04-01...
  Fetching 2019-04-01 → 2019-05-01...
  Fetching 2019-05-01 → 2019-06-01...
  Fetching 2019-06-01 → 2019-07-01...
  Fetching 2019-07-01 → 2019-08-01...
  Fetching 2019-08-01 → 2019-09-01...
  Fetching 2019-09-01 → 2019-10-01...
  Fetching 2019-10-01 → 2019-11-01...
  Fetching 2019-11-01 → 2019-12-01...
  Fetching 2019-12-01 → 2020-01-01...
  Fetching 2020-01-01 → 2020-02-01...
  Fetching 2020-02-01 → 2020-03-01...
  Fetching 2020-03-01 → 2020-04-01...
  Fetching 2020-04-01 → 2020-05-01...
  Fetching 2020-05-01 → 2020-06-01...
  Fetching 2020-06-01 → 2020-07-01...
  Fetching 2020-07-01 → 2020-08-01...
  Fetching 2020-08-01 → 2020-09-01...
  Fetching 2020-09-01 → 2020-10-01...
  Fetching 2020-10-01 → 2020-11-01...
  Fetching 2020-11-01 → 2020-12-01...
  Fetching 2020-12-01 → 2021-01-01...
  Fetching 2021-01-01 → 2021-02-01...
  Fetching 2021-02-01 → 2021-03-01...
  Fetching 2

/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/project/helper_functions_GNN.py:94: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(chunks).sort_index()



Missing values:
price_ee                              0
price_fi                              0
price_lv                              0
price_lt                              0
flow_('ee', 'fi')                     1
flow_('ee', 'lv')                     1
system_production                    18
system_consumption                   21
system_losses                     62113
system_frequency                      1
system_system_balance                15
system_ac_balance                    15
system_production_renewable          19
system_solar_energy_production     8767
dtype: int64


### Mean, std 

In [16]:

df_daily = df_daily.dropna(how="all")

# ==================================================
# ST-GNN: ESTONIAN ENERGY RESILIENCE MODEL
# Target: True energy balance (production + imports - consumption)
# Scenarios: S1 = full grid, S2 = full isolation, S3 = isolated + wind (TBD)
# ==================================================


prices_h = df_prices_daily.copy()
flows_h  = df_flows_daily.copy()
system_h = df_system_daily.copy()

idx      = prices_h.index
flows_h  = flows_h.reindex(idx, method="ffill")
system_h = system_h.reindex(idx, method="ffill")

# True energy balance: production + all imports - consumption
# Positive = surplus, Negative = real deficit even after imports
system_h["gross_supply_input"] = (
    system_h["production"]
    - flows_h[("ee", "fi")] #negative flows are imports, positive flows are exports, so we subtract the imports
    - flows_h[("ee", "lv")] - flows_h[("ee", "ru_narva")] - flows_h[("ee", "ru_pihkva")]
)

print(f"    Mean: {system_h['gross_supply_input'].mean():+.1f} MW")
print(f"    Std:  {system_h['gross_supply_input'].std():.1f} MW")
print(f"    Min:  {system_h['gross_supply_input'].min():+.1f} MW")
print(f"    Max:  {system_h['gross_supply_input'].max():+.1f} MW")
print(f"    True deficit hours: {(system_h['gross_supply_input'] < 0).sum()}")


KeyError: ('ee', 'ru_narva')

EE-FI mean: -624 MW  → Estonia imports ~624 MW from Finland on average

EE-LV mean: +249 MW  → Estonia exports ~249 MW to Latvia on average

In [9]:
flows_h[("ee", "fi")].describe()

count    62113.000000
mean      -624.460111
std        373.506405
min      -1006.658300
25%       -980.483300
50%       -721.183300
75%       -349.475000
max       1033.116000
Name: (ee, fi), dtype: float64

In [11]:
flows_h[("ee", "lv")].describe()

count    62113.000000
mean       248.605089
std        271.094887
min       -820.216700
25%         88.733300
50%        271.600000
75%        441.825000
max        898.666700
Name: (ee, lv), dtype: float64